# 第一步：简单处理 US SEC 报告

本 notebook 对已下载的 SEC HTML filing 做**最基础**的处理，每一步都尽量简单、可读。

流程概览：
1. 定一个要处理的 HTML 文件路径
2. 用 Python 读这个文件
3. 用 BeautifulSoup 解析成「可查询」的结构
4. 从里面取出：标题、文档类型、财年、正文预览
5. 打印出来，确认能拿到这些内容

## 步骤 1：准备路径，选一个 HTML 文件

我们用 `config` 里的 `FILINGS_DIR`，再拼出「某公司 / 某文件名」。这里先用一个**较小的 8-K 文件**做示例，跑得快。

In [ ]:
import os
import sys

# 把当前工作目录加入路径，才能 import config（运行前请 cd 到 02_US_SEC_report）
sys.path.insert(0, os.getcwd())
from config import BASE_DIR, FILINGS_DIR

# 选一个公司文件夹和一个具体的 .htm 文件（这里用 8-K，文件小）
company_folder = "INTEL CORP"
filename = "8-K_2024-01-03_intc-20231229.htm"

file_path = os.path.join(FILINGS_DIR, company_folder, filename)
print("要处理的文件:", file_path)
print("文件存在?", os.path.exists(file_path))

## 步骤 2：读文件内容

用 `open` 读成字符串。SEC 的 HTML 一般是 UTF-8，我们显式指定编码。

In [ ]:
with open(file_path, "r", encoding="utf-8") as f:
    html_content = f.read()

print("读到的字符数:", len(html_content))
print("前 200 个字符:")
print(html_content[:200])

## 步骤 3：用 BeautifulSoup 解析

把 HTML 字符串交给 BeautifulSoup，得到 `soup` 对象。之后就可以用 `soup.title`、`soup.find(...)` 等方式取内容。

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_content, "html.parser")
print("解析完成。类型:", type(soup))

# 马上试一下：取页面标题
if soup.title:
    print("页面 <title> 内容:", soup.title.string)

## 步骤 4：取元数据（文档类型、财年等）

SEC 的 HTML 里有很多带 `name="dei:DocumentType"`、`name="dei:DocumentFiscalYearFocus"` 等属性的标签，我们根据 `name` 找到标签，再取里面的文字。

In [ ]:
def get_metadata_value(soup, name_contains):
    """在 soup 里找 name 属性包含某字符串的标签，返回其文本。"""
    tag = soup.find(attrs={"name": lambda x: x and name_contains in x})
    if tag:
        return tag.get_text(strip=True)
    return None

doc_type = get_metadata_value(soup, "dei:DocumentType")
fiscal_year = get_metadata_value(soup, "dei:DocumentFiscalYearFocus")
period_end = get_metadata_value(soup, "dei:DocumentPeriodEndDate")

print("文档类型:", doc_type)
print("财年:", fiscal_year)
print("报告期末:", period_end)

## 步骤 5：取正文预览

取 `<body>` 里的纯文本，去掉标签。这里只取前 500 个字符，看看内容长什么样。

In [ ]:
body = soup.find("body")
if body:
    full_text = body.get_text(separator=" ", strip=True)
    preview = full_text[:500]
    print("正文总字符数:", len(full_text))
    print("正文预览（前 500 字）:")
    print(preview)
else:
    print("没有找到 body")

## 小结

到这一步我们完成了：
- 读本地 HTML 文件
- 用 BeautifulSoup 解析
- 取标题、文档类型、财年、报告期末
- 取 body 正文并做了简单预览

后续可以在此基础上：按 10-K 的 Item 分节、清洗页眉页脚、再落库等。